# Recurring training availability: the real flow

This notebook follows a domain value through Odysseus's application and in-memory storage:

```text
Domain value
    |
    v
TrainingAvailabilityUseCases
    |
    v
TrainingAvailabilityStore
    ^
    |
InMemoryTrainingAvailabilityStore
```

- **Domain:** what availability means and what is valid.
- **Application:** what Odysseus does with availability.
- **Protocol / contract:** the capability the application requires.
- **Adapter:** a concrete implementation of that capability.

Run the cells in order using the Python 3.12 environment for `apps/Odysseus`, with the project installed. Imports use the installed `src/odysseus` package, so they do not depend on the notebook's working directory.

## Imports

All imports below come from production code.

In [ ]:
from odysseus.adapters.in_memory.availability import InMemoryTrainingAvailabilityStore
from odysseus.application.availability import TrainingAvailabilityUseCases
from odysseus.application.contracts import TrainingAvailabilityStore
from odysseus.domain.availability import (
    DailyTrainingAvailability,
    RecurringTrainingAvailability,
    Weekday,
)

## Build an initial recurring week

Every weekday appears once. Monday offers AM training, Tuesday offers PM training, and Wednesday offers both. Thursday is ordinarily unavailable; Sunday is an explicit rest day. The domain constructors enforce validity, including that rest days cannot also offer training.

In [ ]:
initial_week = RecurringTrainingAvailability(
    days=(
        DailyTrainingAvailability(
            weekday=Weekday.MONDAY,
            am_available=True,
            pm_available=False,
            is_rest_day=False,
        ),
        DailyTrainingAvailability(
            weekday=Weekday.TUESDAY,
            am_available=False,
            pm_available=True,
            is_rest_day=False,
        ),
        DailyTrainingAvailability(
            weekday=Weekday.WEDNESDAY,
            am_available=True,
            pm_available=True,
            is_rest_day=False,
        ),
        DailyTrainingAvailability(
            weekday=Weekday.THURSDAY,
            am_available=False,
            pm_available=False,
            is_rest_day=False,
        ),
        DailyTrainingAvailability(
            weekday=Weekday.FRIDAY,
            am_available=True,
            pm_available=False,
            is_rest_day=False,
        ),
        DailyTrainingAvailability(
            weekday=Weekday.SATURDAY,
            am_available=False,
            pm_available=True,
            is_rest_day=False,
        ),
        DailyTrainingAvailability(
            weekday=Weekday.SUNDAY,
            am_available=False,
            pm_available=False,
            is_rest_day=True,
        ),
    )
)

In [ ]:
print(f"{'Weekday':<10} {'AM':<6} {'PM':<6} Rest")
for day in initial_week.days:
    print(
        f"{day.weekday.value:<10} {str(day.am_available):<6} "
        f"{str(day.pm_available):<6} {day.is_rest_day}"
    )

## Create the adapter and application use case

The constructor requires an existing week. The variable below is annotated with the application-owned `TrainingAvailabilityStore` contract; its runtime object is an `InMemoryTrainingAvailabilityStore`.

The use case depends on the contract's `load()` and `save()` methods. The adapter supplies those methods structurally, without inheriting from the Protocol.

In [ ]:
store: TrainingAvailabilityStore = InMemoryTrainingAvailabilityStore(initial_week)
use_cases = TrainingAvailabilityUseCases(store)

## Retrieve availability through the application

The caller uses `get_recurring_availability()`. The application delegates loading to its store; the caller does not talk directly to the adapter.

In [ ]:
current_week = use_cases.get_recurring_availability()

print(f"{'Weekday':<10} {'AM':<6} {'PM':<6} Rest")
for day in current_week.days:
    print(
        f"{day.weekday.value:<10} {str(day.am_available):<6} "
        f"{str(day.pm_available):<6} {day.is_rest_day}"
    )

## Update the week through the application

Change Monday from AM availability to PM availability. The other six immutable daily values can be reused in a new complete week; the original week remains unchanged. Domain construction validates the new value before the application receives it.

In [ ]:
updated_week = RecurringTrainingAvailability(
    days=(
        DailyTrainingAvailability(
            weekday=Weekday.MONDAY,
            am_available=False,
            pm_available=True,
            is_rest_day=False,
        ),
        *initial_week.days[1:],
    )
)

Save through the application, then retrieve through the application again. The in-memory adapter replaces the value it holds.

In [ ]:
use_cases.update_recurring_availability(updated_week)
reloaded_week = use_cases.get_recurring_availability()

print(f"{'Weekday':<10} {'AM':<6} {'PM':<6} Rest")
for day in reloaded_week.days:
    print(
        f"{day.weekday.value:<10} {str(day.am_available):<6} "
        f"{str(day.pm_available):<6} {day.is_rest_day}"
    )

## Architecture recap

```text
Caller
  |
  v
TrainingAvailabilityUseCases
  |
  v
TrainingAvailabilityStore Protocol
  ^
  |
InMemoryTrainingAvailabilityStore
  |
  v
RecurringTrainingAvailability
```

The domain owns validity. The application owns the retrieval and update use cases. Its Protocol defines the required storage capability, and the adapter provides that capability while holding the domain value.

The upward arrow shows that the adapter satisfies the contract; the Protocol is not an extra runtime object. Replacing the in-memory adapter later with another implementation of the same contract should not require changing the use case.